# 06. 지원 우선순위 (12번 ③)

**질문:** 비 피해를 기준으로, 어느 지역·업종을 먼저 지원해야 하나?

- 단위: 지역(춘천·강남) × 업종 그룹 7개 = 14칸 ("상권 = 시·군·구 × 업종", 팀 가정)
- 코드: `src/priority.py` · 업종 연결표: `data/processed/업종연결표.csv` · 상태: **팀 검증 전**

## 이 셀이 하는 일: 실행 위치와 환경 확인

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':   # notebooks 폴더에서 실행된 경우
    os.chdir('..')                                   # 프로젝트 폴더로 이동
sys.path.insert(0, 'src')

import numpy, pandas, statsmodels
print('실행 폴더:', os.path.basename(os.getcwd()))
print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| statsmodels', statsmodels.__version__)

실행 폴더: bigcontest_team
Python 3.14.7 | numpy 2.5.2 | pandas 3.0.5 | statsmodels 0.15.0


## 점수 만드는 법 (쉬운 설명)

| 지표 | 뜻 | 가중치 |
|---|---|---|
| ① 점포당 기대 손실 | 비 한 번에 사라진 결제액 × 한 시즌(7~12월) 평균 호우 횟수(30년) ÷ 점포 수. **단, 사라진 결제액의 90% 구간이 0을 포함하면 0** | 40 |
| ② 순손실률 N5 | 비 한 번에 사라지고 5일 안에 돌아오지 않은 결제 비율 | 20 |
| ③ 외지인 의존도 | 결제 중 외지인 비중 (지역 안에서만 비교) | 20 |
| ④ 확실성 | N5의 90% 구간 아래쪽 끝이 0보다 크면 1 | 20 |

1. 지표마다 14칸 안에서 순위를 매겨 **0~1 점**으로 바꿉니다 (1등 1.0, 꼴찌 0). 단위가 달라도 같은 눈금으로 더하기 위해서입니다.
2. 가중치를 곱해 더하면 0~100점 → **70점 이상 지원 1순위, 40~70점 2순위, 40점 미만 관찰**.
3. 가중치·기준을 바꿔도 순위가 유지되는지 확인합니다(민감도).

- 결제액은 **신한카드 외지인 결제분**이고, 택시는 뺐습니다(건당 금액 이상).
- ①의 "구간이 0을 포함하면 0" 규칙은 11번 호우 후속 결정(구간이 0을 포함하는 금액은 쓰지 않음)을 따른 것입니다. 원래 값은 `점포당기대손실_원값` 열에 있습니다.
- ③ 외지인의 뜻: 춘천은 강원 밖 거주자(관광객에 가까움), 강남은 서울 밖 거주자(출퇴근자로 추정) → 그래서 지역 안에서만 순위를 매깁니다.

## 이 셀이 하는 일: 데이터 읽기와 전체 계산 (약 40초)

In [2]:
import pandas as pd
from event_study import load_all
from priority import run, load_mapping

pd.set_option('display.width', 250)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
data = load_all()
ind, sc, ranks, summ, stores, events = run(data)
print('시즌당 호우 사건(1995~2024년 7~12월 평균):', events)

시즌당 호우 사건(1995~2024년 7~12월 평균): {'춘천': 7.2, '강남': 7.7}


## 업종 연결표 요약 (상가정보 소분류 → 카드 그룹)

In [3]:
m = load_mapping()
print('소분류', len(m), '개')
pd.crosstab(m['카드그룹'], m['확실도'])

소분류 247 개


확실도,낮음,높음,보통
카드그룹,,,
교통·숙박,2,6,5
상품쇼핑,0,1,39
생활·교육서비스,0,18,29
식료품,2,12,5
여가,1,20,6
외식,0,39,0
의료·약국,1,20,0
제외,0,41,0


In [4]:
# 확실도가 '낮음'인 연결 (팀 확인 필요)
m[m['확실도'] == '낮음'][['대분류', '중분류', '소분류', '카드그룹', '메모']]

,대분류,중분류,소분류,카드그룹,메모
75,소매,담배 소매,담배/전자담배 소매업,식료품,NaN
110,소매,오락용품 소매,자전거 소매업,여가,NaN
117,소매,의약·화장품 소매,의료기기 소매업,의료·약국,NaN
177,시설관리·임대,운송장비 대여,기타 운송장비 대여업,교통·숙박,NaN
178,시설관리·임대,운송장비 대여,자동차 대여업,교통·숙박,NaN
208,음식,기타 간이,떡/한과,식료품,카드에서는 제과점·기타식품에 가까움


## 점포 수 (2025년 10월 기준, 연결표 적용)

In [5]:
stores

,춘천,강남
상권업종소분류명,,
식료품,1557,2696
외식,5270,11488
의료·약국,588,3755
교통·숙박,975,1207
여가,1060,2537
상품쇼핑,2271,6548
생활·교육서비스,3998,17714


## 지표 14칸

In [6]:
ind[['지역', '그룹', '사건당손실(만원)', '사건당손실_5%(만원)', '금액확인', '점포수',
     '점포당기대손실_원값', '점포당기대손실', 'N5', 'N5_5%', 'N5_95%', '외지인비중', '확실성']]

,지역,그룹,사건당손실(만원),사건당손실_5%(만원),금액확인,점포수,점포당기대손실_원값,점포당기대손실,N5,N5_5%,N5_95%,외지인비중,확실성
0,춘천,식료품,"13,799.38","6,140.28",True,1557,"638,121.60","638,121.60",47.16,20.98,72.73,16.08,1.00
1,춘천,외식,"20,359.05","5,305.40",True,5270,"278,150.23","278,150.23",41.56,10.83,68.57,22.01,1.00
2,춘천,의료·약국,"3,420.02",668.80,True,588,"418,778.52","418,778.52",30.78,6.02,53.80,7.80,1.00
3,춘천,교통·숙박,"13,206.24","6,770.21",True,975,"975,230.18","975,230.18",78.74,40.37,116.09,16.49,1.00
4,춘천,여가,"20,187.77","9,904.13",True,1060,"1,371,244.47","1,371,244.47",79.25,38.88,117.64,32.04,1.00
5,춘천,상품쇼핑,"4,051.28",-239.04,False,2271,"128,442.11",0.00,49.67,-2.93,97.10,16.93,0.00
6,춘천,생활·교육서비스,945.76,-55.05,False,3998,"17,032.29",0.00,18.06,-1.05,37.21,8.63,0.00
7,강남,식료품,"8,074.05","-62,069.65",False,2696,"230,601.58",0.00,1.71,-13.12,15.63,55.89,0.00
8,강남,외식,"-10,183.61","-51,135.48",False,11488,"-68,257.12",0.00,-2.52,-12.63,7.15,45.54,0.00
9,강남,의료·약국,"-77,731.06","-172,064.27",False,3755,"-1,593,952.40",0.00,-9.78,-21.66,1.69,32.66,0.00


## 결과 1. 점수와 등급 (기본안)

In [7]:
sc

,지역,그룹,p_점포당기대손실,p_N5,p_외지인비중,p_확실성,점수,순위,등급
0,춘천,여가,1.00,1.00,1.00,0.81,96.15,1,지원 1순위
1,춘천,교통·숙박,0.92,0.92,0.50,0.81,81.54,2,지원 1순위
2,강남,교통·숙박,0.85,0.46,0.67,0.81,72.56,3,지원 1순위
3,춘천,외식,0.62,0.69,0.83,0.81,71.28,4,지원 1순위
4,춘천,식료품,0.77,0.77,0.33,0.81,68.97,5,지원 2순위
5,춘천,의료·약국,0.69,0.62,0.00,0.81,56.15,6,지원 2순위
6,춘천,상품쇼핑,0.27,0.85,0.67,0.27,46.41,7,지원 2순위
7,강남,상품쇼핑,0.27,0.38,1.00,0.27,43.85,8,지원 2순위
8,강남,식료품,0.27,0.31,0.83,0.27,38.97,9,관찰
9,강남,외식,0.27,0.23,0.50,0.27,30.77,10,관찰


## 결과 2. 민감도: 시나리오별 순위

In [8]:
ranks

기본안 (40/20/20/20)  같은 가중치 (25×4)  점포당기대손실 빼기  N5 빼기  외지인비중 빼기  확실성 빼기  불확실한 금액도 그대로 (옛 기본안)  점포 수 2025-06 기준  점포 수 2025-12 기준
지역 그룹                                                                                                                                     
춘천 여가                        1              1           1      1         1       1                     1                1                1
   교통·숙박                     2              2           3      3         2       2                     2                2                2
강남 교통·숙박                     3              4           4      2         4       3                     4                3                3
춘천 외식                        4              3           2      4         6       4                     5                4                4
   식료품                       5              5           5      5         3       5                     6                5                5
   의료·약국                     6              6           8      6         5       7                     7                6                6
   상품쇼핑                      7              7           6      9         7       6                     8                7                7
강남 상품쇼핑                      8              8           7      7         9       8                     3                8                8
   식료품                       9              9           9      8        10       9                     9                9                9
   외식                       10             10          10     10        11      10                    11               10               10
춘천 생활·교육서비스                 11             11          11     12         8      11                    10               11               11
강남 여가                       12             12          12     11        13      12                    12               12               12
   생활·교육서비스                 13             13          13     12        14      13                    13               13               13
   의료·약국                    14             14          14     14        12      14                    14               14               14

In [9]:
summ

,시나리오,기본안 상위4와 겹침,순위 상관(스피어만)
0,기본안 (40/20/20/20),4/4,1.00
1,같은 가중치 (25×4),4/4,1.00
2,점포당기대손실 빼기,4/4,0.97
3,N5 빼기,4/4,0.97
4,외지인비중 빼기,3/4,0.94
5,확실성 빼기,4/4,1.00
6,불확실한 금액도 그대로 (옛 기본안),3/4,0.93
7,점포 수 2025-06 기준,4/4,1.00
8,점포 수 2025-12 기준,4/4,1.00


## 읽을 때 주의할 점
- **기본안은 "불확실한 금액은 0" 규칙을 씁니다 (2026-10-21 변경). 이 규칙은 결과를 본 뒤 바꾼 것입니다.** 처음 기본안은 금액 점추정을 그대로 써서 강남 상품쇼핑이 3위였지만, 그 금액의 90% 구간이 −17.7억~+47.0억 원으로 0을 크게 포함합니다. 근거는 11번 호우 후속 작업 기록의 "강남은 구간이 0을 포함해 금액 환산 결과로 쓰지 않음"이지만, 바꾼 시점이 결과를 본 뒤이므로 보고서에도 그렇게 밝힙니다. 옛 기본안은 민감도 시나리오 "불확실한 금액도 그대로 (옛 기본안)"로 남겨 두었습니다.
- **춘천 여가·교통·숙박은 모든 시나리오에서 1~4위**입니다. 가중치·점포 기준 시점을 바꿔도 흔들리지 않는 결과입니다.
- **교통·숙박의 점포 수는 숙박·주유소·여행사만 잡힌 것**입니다. 택시·버스·주차장은 상가정보에 점포가 없어서 점포당 손실이 커 보일 수 있습니다.
- **강남의 음수 손실**(외식·여가·의료·생활)은 "비 온 날 오히려 늘었다"가 아니라, 대부분 90% 구간이 0을 포함하는 "차이가 확인되지 않음"입니다.
- 가중치 40/20/20/20은 제안값이며 근거가 있는 값이 아닙니다. 팀이 정해야 합니다.
- 상관관계를 보는 분석입니다. 순위는 "비 피해와 관련이 큰 순서"로 읽습니다.